# Searchlight RSA boilerplate

In [1]:
import os
import numpy as np
from scipy.stats import gamma
import subprocess
import nibabel as nib
from pathlib import Path
import psutil
import scipy.io as sio
from scipy.stats import spearmanr, pearsonr, kendalltau
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
# Add Connectome Workbench to PATH
os.environ['PATH'] = '/opt/workbench/bin_linux64:' + os.environ['PATH']
timing_df = pd.read_csv("/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv")
timing_excel = pd.read_excel("/home/amin/Research/Representation/Movie/data/Setareh/Data/TOI_V3.xlsx")

In [2]:
def preprocess_fmri(fmri_data, timing_df, TR, bin_sec, delay_sec, drop_last=True):
    """
    Extract and bin fMRI data with BOLD delay compensation.
    """
    n_vertices, n_timepoints = fmri_data.shape
    segments = []
    
    for _, row in timing_df.iterrows():
        # --- FIX: Removed trailing commas below ---
        onset_sec = row["onset_sec"]
        duration_sec = row["duration_sec"]

        # Apply delay to the stimulus window
        onset_d = onset_sec + delay_sec
        end_d = onset_sec + duration_sec + delay_sec

        # Convert to timepoint indices
        start_tp = int(np.floor(onset_d / TR)) 
        end_tp = int(np.ceil(end_d / TR))

        # Clip to valid range
        start_tp = max(start_tp, 0)
        end_tp = min(end_tp, n_timepoints)

        # Handle edge case where delay pushes segment out of bounds
        if end_tp <= start_tp:
            raise RuntimeError(f'Segmentation error from {onset_sec} to {onset_sec+duration_sec}') 

        segment = fmri_data[:, start_tp:end_tp]

        # Timepoints per bin
        tp_per_bin = int(np.floor(bin_sec / TR))
        if tp_per_bin < 1:
            raise ValueError(f"bin_sec/TR is < 1 (bin_sec={bin_sec}, TR={TR})")

        # Drop leftover timepoints that don't fill a full bin
        if segment.shape[1] == 0:
            continue
            
        if drop_last:
            n_bins = int(np.floor(segment.shape[1] / tp_per_bin))
        else:
            n_bins = int(np.ceil(segment.shape[1] / tp_per_bin))
            
        # Ensure we have enough data for at least one bin
        if n_bins == 0:
            continue
            
        # Truncate to exact multiple of bin size to allow reshape
        # If not dropping last, we might need padding, but standard approach here 
        # usually implies we only take what fits or we strictly reshape.
        # Given your code used reshape(), we must ensure exact size.
        valid_length = n_bins * tp_per_bin
        
        # If strict floor was used, this just trims tail
        if segment.shape[1] >= valid_length:
            segment = segment[:, :valid_length]
        else:
            # If we calculated bins via ceil but don't have enough data (rare), skip
            continue

        binned = segment.reshape(n_vertices, n_bins, tp_per_bin).mean(axis=2)
        segments.append(binned)
        # print(f"  Video {row['video_id']}: {binned.shape[1]} bins")
    
    if len(segments) == 0:
        return np.zeros((n_vertices, 0), dtype=fmri_data.dtype)
        
    filtered_fmri = np.concatenate(segments, axis=1)
    return filtered_fmri

In [3]:
def compute_geodesic_distance_matrix(surface_file, output_dconn, 
                                     limit_mm=None, roi_metric=None,
                                     corrected_areas=None, naive=False):
    """
    Compute full geodesic distance matrix using wb_command, skipping if exists.
    """
    print(f"Checking for geodesic distance matrix...")
    print(f"  Target Output: {output_dconn}")
    
    # --- CHECK 1: If file exists, skip computation ---
    if os.path.exists(output_dconn):
        print(f"  [FOUND] Distance matrix already exists. Skipping computation.")
        return output_dconn
    
    print(f"  [NOT FOUND] Computing all-to-all geodesic distances...")
    print(f"  Surface: {surface_file}")

    # Build command
    cmd = [
        'wb_command',
        '-surface-geodesic-distance-all-to-all',
        surface_file,
        output_dconn
    ]
    
    if roi_metric is not None:
        cmd.extend(['-roi', roi_metric])
    if limit_mm is not None:
        cmd.extend(['-limit', str(limit_mm)])
    if corrected_areas is not None:
        cmd.extend(['-corrected-areas', corrected_areas])
    if naive:
        cmd.append('-naive')
    
    # Run command
    print("  Running wb_command...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"wb_command failed: {result.stderr}")
    
    print(f"  Success! Saved to {output_dconn}")
    return output_dconn


def get_k_nearest_neighbors(distance_matrix_path, output_neighbors_file, k=100):
    """
    Get K nearest neighbors. 
    1. Checks if 'output_neighbors_file' exists. If so, loads and returns it.
    2. If not, loads the dconn matrix, computes neighbors, saves, and returns.
    
    Parameters:
    -----------
    distance_matrix_path : str
        Path to the dconn/npy distance matrix (only loaded if calculation is needed).
    output_neighbors_file : str
        Path where the neighbor indices (.npy) should be saved/loaded from.
    k : int
        Number of nearest neighbors.
    """
    print(f"Checking for {k}-nearest neighbors file...")
    print(f"  Target Output: {output_neighbors_file}")

    # --- CHECK 2: If neighbors file exists, load and return ---
    if os.path.exists(output_neighbors_file):
        print(f"  [FOUND] Neighbors file exists. Loading from disk...")
        neighbor_indices = np.load(output_neighbors_file)
        
        # Safety check on shape
        if neighbor_indices.shape[1] != k:
            print(f"  WARNING: Existing file has k={neighbor_indices.shape[1]}, but requested k={k}. Recomputing...")
        else:
            return neighbor_indices

    # --- COMPUTATION: Only happens if check failed ---
    print(f"  [NOT FOUND] Computing nearest neighbors...")
    
    # Load the heavy distance matrix only now
    if isinstance(distance_matrix_path, (str, Path)):
        print(f"  Loading distance matrix from {distance_matrix_path}...")
        if str(distance_matrix_path).endswith('.dconn.nii'):
            # Load CIFTI
            img = nib.load(distance_matrix_path)
            distance_matrix = img.get_fdata()
        elif str(distance_matrix_path).endswith('.npy'):
            distance_matrix = np.load(distance_matrix_path)
        else:
            raise ValueError("Unknown distance matrix format. Use .dconn.nii or .npy")
    else:
        distance_matrix = distance_matrix_path

    n_vertices = distance_matrix.shape[0]
    print(f"  Finding {k} neighbors for {n_vertices} vertices...")
    
    # Pre-allocate
    neighbor_indices = np.zeros((n_vertices, k), dtype=np.int32)
    
    # Compute
    # Note: argsort on a huge matrix is slow. 
    # Optimization: argpartition is O(n) instead of O(n log n) for top K
    for i in range(n_vertices):
        if i % 5000 == 0:
            print(f"  Progress: {i}/{n_vertices}")
        
        # Fast way to get top K smallest indices (unsorted)
        idx = np.argpartition(distance_matrix[i, :], k)[:k]
        
        # If you need them sorted by distance strictly:
        # idx = idx[np.argsort(distance_matrix[i, idx])]
        
        neighbor_indices[i, :] = idx
    
    print(f"  Saving neighbors to {output_neighbors_file}...")
    np.save(output_neighbors_file, neighbor_indices)
    
    return neighbor_indices

In [4]:
def spm_hrf(tr, oversampling=16):
    dt = tr / oversampling
    u = np.arange(0, 32, dt) - dt/2
    h = (6 * gamma.pdf(u, 6, loc=0, scale=1) - 
         1/6 * gamma.pdf(u, 16, loc=0, scale=1))
    return h / np.sum(h)

def process_model_embeddings(model_embeddings, bin_sec, timing_df, hrf=True, normalization=True):
    """
    Handles HRF convolution and PER-RUN normalization/demeaning.
    """
    # 1. Setup HRF
    hrf_kernel = spm_hrf(bin_sec, oversampling=1)
    
    # 2. Slice into individual video segments
    video_bin_counts = (timing_df['duration_sec'] / bin_sec).astype(int).values
    segments = []
    current_idx = 0
    
    for n_bins in video_bin_counts:
        segment = model_embeddings[current_idx : current_idx + n_bins, :]
        
        # Apply HRF if requested
        if hrf:
            conv_segment = np.zeros_like(segment)
            for i in range(segment.shape[1]):
                conv = np.convolve(segment[:, i], hrf_kernel, mode='full')
                conv_segment[:, i] = conv[:n_bins]
            segments.append(conv_segment)
        else:
            segments.append(segment)
            
        current_idx += n_bins

    # 3. Perform Per-Run Normalization OR Demeaning
    print(f"Performing Per-Run {'Normalization (Z-score)' if normalization else 'Demeaning (Mean centering)'}...")
    normalized_segments = [None] * len(segments)
    unique_runs = timing_df['run_id'].unique()
    
    for rid in unique_runs:
        run_indices = timing_df.index[timing_df['run_id'] == rid].tolist()
        run_data = np.concatenate([segments[i] for i in run_indices], axis=0)
        
        if normalization:
            # Z-score the entire run block
            run_proc = zscore(run_data, axis=0)
            run_proc = np.nan_to_num(run_proc) # Clean up constant features
        else:
            # Demean per run (Subtract the mean of the run from the run data)
            run_proc = run_data - np.mean(run_data, axis=0)
        
        # Slice the processed run block back into original segments
        slice_ptr = 0
        for i in run_indices:
            v_len = video_bin_counts[i]
            normalized_segments[i] = run_proc[slice_ptr : slice_ptr + v_len, :]
            slice_ptr += v_len
    
    segments = normalized_segments

    # 4. Re-concatenate into final embedding matrix
    return np.concatenate(segments, axis=0)



def save_direct_cifti(left_data, right_data, template_cifti_path, output_path, metric_name="RSA_Metric"):
    """
    Creates a CIFTI file directly by hijacking the header of an existing HCP template.
    
    Parameters:
    -----------
    left_data : np.ndarray
        1D array of left hemisphere values (e.g., shape 32492,)
    right_data : np.ndarray
        1D array of right hemisphere values (e.g., shape 32492,)
    template_cifti_path : str
        Path to ANY existing .dscalar.nii file in HCP 32k space to use as a header template.
    output_path : str
        Where to save the new .dscalar.nii file.
    """
    print(f"\n  [CIFTI] Generating direct CIFTI using template header...")
    
    # 1. Concatenate the data (Left must come before Right in standard HCP CIFTI)
    # Ensure any NaNs are handled (e.g., converted to 0 for viz) if desired
    left_clean = np.nan_to_num(left_data, nan=0.0)
    right_clean = np.nan_to_num(right_data, nan=0.0)
    
    combined_data = np.hstack((left_clean, right_clean))
    
    # CIFTI dense scalars expect shape (1, n_vertices) or (n_maps, n_vertices)
    combined_data = combined_data.reshape(1, -1) 
    
    # 2. Load the template to steal its geometry mappings
    template_img = nib.load(template_cifti_path)
    
    # Verify the vertex count matches (e.g., 64984 for full 32k, or ~59k if medial wall is already masked out in the template)
    if combined_data.shape[1] != template_img.shape[1]:
        raise ValueError(
            f"Shape mismatch! Your concatenated data has {combined_data.shape[1]} vertices, "
            f"but the template expects {template_img.shape[1]}. "
            f"Ensure your template handles the medial wall the same way your arrays do."
        )

    # 3. Create the new CIFTI image using the hijacked header
    cifti = nib.Cifti2Image(
        combined_data.astype(np.float32), 
        header=template_img.header,
        nifti_header=template_img.nifti_header
    )
    
    # Optional: Update the map name inside the XML so Workbench labels it correctly
    try:
        scalar_axis = cifti.header.get_axis(0)
        scalar_axis.name[0] = metric_name
    except Exception as e:
        print(f"    (Note: Could not update internal map name: {e})")

    # 4. Save to disk
    nib.save(cifti, output_path)
    print(f"  [SUCCESS] Saved direct CIFTI: {output_path}")

from joblib import Parallel, delayed


def _single_searchlight(vertex_idx, neighbors, fmri_data, model_rdm_vec,
                         limits, valid_indices_mask, block_diagonal, 
                         method, min_variance):
    try:
        searchlight_activations = fmri_data[neighbors, :].T

        if searchlight_activations.std() < min_variance:
            return vertex_idx, np.nan, np.nan, 'skipped'

        if block_diagonal:
            brain_vecs = []
            for idx, (start, end) in enumerate(limits):
                if not valid_indices_mask[idx]:
                    continue
                vid_brain = searchlight_activations[start:end, :]
                brain_vecs.append(pdist(vid_brain, metric='correlation'))
            if not brain_vecs:
                return vertex_idx, np.nan, np.nan, 'skipped'
            searchlight_rdm_vec = np.hstack(brain_vecs)
        else:
            searchlight_rdm_vec = pdist(searchlight_activations, metric='correlation')

        if len(searchlight_rdm_vec) != len(model_rdm_vec):
            return vertex_idx, np.nan, np.nan, 'failed'

        if searchlight_rdm_vec.std() < min_variance:
            return vertex_idx, np.nan, np.nan, 'skipped'

        if method == 'spearman':
            corr, pval = spearmanr(searchlight_rdm_vec, model_rdm_vec, nan_policy='omit')
        elif method == 'pearson':
            corr, pval = pearsonr(searchlight_rdm_vec, model_rdm_vec)
        elif method == 'kendall':
            corr, pval = kendalltau(searchlight_rdm_vec, model_rdm_vec, nan_policy='omit')
        else:
            raise ValueError(f"Unknown method: {method}")

        if np.isnan(corr) or np.isnan(pval):
            return vertex_idx, np.nan, np.nan, 'failed'

        return vertex_idx, corr, pval, 'computed'

    except Exception:
        return vertex_idx, np.nan, np.nan, 'failed'


def compute_searchlight_rsa_map(fmri_data, neighbor_indices, model_data,
                                timing_df, bin_sec, block_diagonal=False, 
                                output_file=None, normalization=True, hrf=True, method='spearman',
                                save_pvalues=True, min_variance=1e-10):
    """
    Compute RSA correlation map for all searchlights, supporting global and block-diagonal RDM calculation.
    """
    
    n_vertices, n_timepoints = fmri_data.shape
    print(f"fMRI data shape: {n_vertices} vertices × {n_timepoints} timepoints")
    
    # -------------------------------------------------------------------------
    # 1. LOAD AND PREPROCESS MODEL
    # -------------------------------------------------------------------------
    if isinstance(model_data, (str, Path)):
        raw_model = np.load(model_data)
    else:
        raw_model = model_data

    clean_model = process_model_embeddings(
            raw_model, 
            bin_sec=bin_sec, 
            timing_df=timing_df, 
            hrf=hrf, 
            normalization=normalization
        )
    
    if clean_model.shape[0] != n_timepoints:
        raise ValueError(
            f"Model timepoints ({clean_model.shape[0]}) doesn't match "
            f"fMRI timepoints ({n_timepoints})"
        )

    # -------------------------------------------------------------------------
    # 2. PREPARE MODEL VECTOR (Global vs Block-Diagonal)
    # -------------------------------------------------------------------------
    video_bins = (timing_df['duration_sec'] / bin_sec).astype(int).values
    limits = []
    current_idx = 0
    for b in video_bins:
        limits.append((current_idx, current_idx + b))
        current_idx += b

    if block_diagonal:
        print("\n--- STRATEGY: BLOCK-DIAGONAL (WITHIN-VIDEO) RSA ---")
        model_vecs = []
        valid_indices_mask = []
        
        for i, (start, end) in enumerate(limits):
            vid_model = clean_model[start:end, :]
            
            # Skip short segments
            if vid_model.shape[0] < 3: 
                valid_indices_mask.append(False)
                continue
            
            valid_indices_mask.append(True)
            model_vecs.append(pdist(vid_model, metric='correlation'))
            
        model_rdm_vec = np.hstack(model_vecs)
        
    else:
        print("\n--- STRATEGY: GLOBAL RSA ---")
        model_rdm_vec = pdist(clean_model, metric='correlation')

    print(f"Final Model Vector elements: {len(model_rdm_vec)}")
    
    if model_rdm_vec.std() < min_variance:
        raise ValueError("Model RDM has zero variance! Cannot compute correlations.")
    
    # -------------------------------------------------------------------------
    # 3. COMPUTE RSA FOR SEARCHLIGHTS
    # -------------------------------------------------------------------------
    n_searchlights = neighbor_indices.shape[0]
    rsa_correlations = np.full(n_searchlights, np.nan, dtype=np.float32)
    
    if save_pvalues:
        rsa_pvalues = np.full(n_searchlights, np.nan, dtype=np.float32)
    
    print(f"\nComputing RSA for {n_searchlights} searchlights...")
    print(f"RSA comparison method: {method}")

    results = Parallel(n_jobs=16, prefer='processes', verbose=1)(
        delayed(_single_searchlight)(
            i, neighbor_indices[i, :], fmri_data,
            model_rdm_vec, limits,
            valid_indices_mask if block_diagonal else [],
            block_diagonal, method, min_variance
        )
        for i in range(n_searchlights)
    )

    n_computed = 0
    n_skipped = 0
    n_failed = 0
    searchlight_results_list = []

    for vertex_idx, corr, pval, status in results:
        if status == 'computed':
            rsa_correlations[vertex_idx] = corr
            if save_pvalues:
                rsa_pvalues[vertex_idx] = pval
            searchlight_results_list.append({
                'vertex': vertex_idx,
                'correlation': corr,
                'pvalue': pval,
            })
            n_computed += 1
        elif status == 'skipped':
            n_skipped += 1
        else:
            n_failed += 1
    
    print(f"\n\nRSA computation complete!")
    print(f"  Successfully computed: {n_computed} / {n_searchlights} ({100*n_computed/n_searchlights:.1f}%)")
    print(f"  Skipped (low variance): {n_skipped} ({100*n_skipped/n_searchlights:.1f}%)")
    print(f"  Failed (errors): {n_failed} ({100*n_failed/n_searchlights:.1f}%)")
    
    # -------------------------------------------------------------------------
    # 4. SAVE RESULTS
    # -------------------------------------------------------------------------
    if output_file:
        output_path = Path(output_file)
        
        np.save(output_path.with_suffix('.npy'), rsa_correlations)
        if save_pvalues:
            np.save(output_path.with_name(output_path.stem + '_pvalues.npy'), rsa_pvalues)
    return rsa_correlations

In [5]:
def complete_searchlight_rsa_pipeline(hemisphere, label, surface_file, fmri_data, 
                                      timing_df, bin_sec, model_data, 
                                      output_dir, data_dir, n_neighbors, 
                                      normalization, hrf, method, 
                                      block_diagonal, limit_mm=None):
    """
    Complete memory-efficient Searchlight RSA pipeline.
    
    Parameters:
    -----------
    hemisphere: str
        'left' or 'right'.
    label: str
        Label name for model-modality (e.g., 'peav_small_16f-video').
    surface_file : str
        Surface geometry file (.surf.gii).
    fmri_data : np.ndarray
        Preprocessed fMRI data (n_vertices, n_timepoints).
    timing_df: pd.DataFrame
        Timing information for each video.
    bin_sec : float
        Bin size in seconds.
    model_data : str or np.ndarray
        Model RDM or embeddings (.npy file).
    output_dir : str
        Where to save the RSA results.
    data_dir : str or Path
        Directory for pre-computed geometry files.
    n_neighbors : int
        Searchlight size (k).
    normalization : bool
        Whether to z-score the model.
    hrf : bool
        Whether to apply HRF convolution.
    method : str
        'spearman', 'pearson', or 'kendall'.
    block_diagonal : bool
        If True, performs Within-Video RSA.
    limit_mm : float, optional
        Distance limit in mm for neighbors.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    
    mode_str = "BLOCK-DIAGONAL (Within-Video)" if block_diagonal else "GLOBAL"
    
    print("=" * 70)
    print(f"SEARCHLIGHT RSA PIPELINE [{hemisphere.upper()}] - {mode_str}")
    print("=" * 70)
    
    # ---------------------------------------------------------
    # Step 1: Compute/Check Geodesic Distances
    # ---------------------------------------------------------
    print("\nSTEP 1: Checking geodesic distances...")
    dconn_filename = f"geodesic_distances.dconn_{hemisphere}.nii"
    dconn_path = data_dir / dconn_filename
    
    compute_geodesic_distance_matrix(
        surface_file=surface_file,
        output_dconn=str(dconn_path),
        limit_mm=limit_mm
    )
    
    # ---------------------------------------------------------
    # Step 2: Compute/Check Neighbors
    # ---------------------------------------------------------
    print(f"\nSTEP 2: Checking {n_neighbors} nearest neighbors...")
    neighbors_filename = f"neighbor_indices_k{n_neighbors}_{hemisphere}.npy"
    neighbors_path = data_dir / neighbors_filename
    
    neighbor_indices = get_k_nearest_neighbors(
        distance_matrix_path=dconn_path,
        output_neighbors_file=str(neighbors_path),
        k=n_neighbors
    )
    
    # ---------------------------------------------------------
    # Step 3: Compute RSA Map
    # ---------------------------------------------------------
    print("\nSTEP 3: Computing RSA correlation map...")
    
    # Robust Naming Convention
    suffix = "blockdiag" if block_diagonal else "global"
    norm_lbl = "normalized" if normalization else "notnormalized"
    hrf_lbl = "hrf" if hrf else "nohrf"
    
    rsa_filename = f"rsa_{label}_k{n_neighbors}_{method}_{norm_lbl}_{hrf_lbl}_bin{int(bin_sec)}s_{hemisphere}_{suffix}"
    rsa_output = output_dir / rsa_filename
    
    rsa_correlations = compute_searchlight_rsa_map(
        fmri_data=fmri_data,
        neighbor_indices=neighbor_indices,
        model_data=model_data,
        output_file=str(rsa_output),
        timing_df=timing_df,        
        bin_sec=bin_sec,            
        normalization=normalization, 
        hrf=hrf,
        method=method,
        save_pvalues=True,
        min_variance=1e-10,
        block_diagonal=block_diagonal 
    )
    
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE!")
    print("=" * 70)
    print(f"None-Image Results saved in: {output_dir}")
    
    return rsa_correlations

# automated megaloop

In [13]:
import numpy as np
import pandas as pd
import scipy.io as sio
from pathlib import Path
import sys


# ============================================================================
# CONFIGURATION
# ============================================================================

''' 
Based on viewing the results, this configuration was chosen. de-meaning was added to when normalization is not performed. as a result the un-normalized results 
of configs other than 5s delay which didnt use normalization are not the latest version of the code 
(previous version used nothing when normalization is false; current one does demeaning). so the the only cofigurations with the latest variations in code are the ones
who are saved as cifti.
'''

use_normalization = [True , False]
use_hrf = [True , False]
BLOCKDIAG = [True, False]
BIN_SEC_LIST = [2.0, 5.0, 10.0, 1.0]  
DELAY_SEC_LIST = [5.0]
n_neighbors=100

# Paths
template_cifti_path = '/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.curvature_MSMAll.32k_fs_LR.dscalar.nii'
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output")
MAT_FILE_LEFT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
MAT_FILE_RIGHT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"
GLASSER_LEFT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii"
GLASSER_RIGHT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii"
TIMING_CSV = "/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv"

# Models & Modalities
models = {
    "pe-av-small-16-frame"
}
modalities = {
    "joint": "av",
    # "audio": "a",
    # "video": "v"
}
RSA_METHODS = ['spearman']
TR = 1.0  # Assumed TR 

# ============================================================================
# STEP 0: LOAD RAW DATA ONCE (Optimization)
# ============================================================================
print("=" * 70)
print("LOADING RAW DATA (Running Once)")
print("=" * 70)

timing_df = pd.read_csv(TIMING_CSV)

# Load Left fMRI
print("Loading Left Hemisphere...")
mat_data_left = sio.loadmat(MAT_FILE_LEFT)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492: fmri_left_raw = fmri_left_raw.T

# Load Right fMRI
print("Loading Right Hemisphere...")
mat_data_right = sio.loadmat(MAT_FILE_RIGHT)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492: fmri_right_raw = fmri_right_raw.T

# Pre-calculate valid masks (Medial Wall Removal)
valid_mask_left = fmri_left_raw.std(axis=1) > 0
valid_mask_right = fmri_right_raw.std(axis=1) > 0

print(f"Data Loaded. Left Valid: {np.sum(valid_mask_left)}, Right Valid: {np.sum(valid_mask_right)}")
data_dir = Path("/home/amin/Research/Representation/Movie/data/Setareh")
surface_left =  Path("/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii")
surface_right = Path("/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii")

# ============================================================================
# MAIN LOOP
# ============================================================================

for hrf in use_hrf:
    # If HRF is ON, Delay is irrelevant (set to 0). If HRF is OFF, loop over delays.
    current_delays = [0] if hrf else DELAY_SEC_LIST
    
    for delay_sec in current_delays:
        for bin_sec in BIN_SEC_LIST:
            
            # Print once per timescale
            print(f"\n[FMRI] Preprocessing: Bin={bin_sec}s, Delay={delay_sec}s, HRF={hrf}")
            
            # 1. LOAD RAW PREPROCESSED DATA (Base Copy)
            fmri_l_base = preprocess_fmri(fmri_left_raw, timing_df, TR=TR, bin_sec=bin_sec, delay_sec=delay_sec)
            fmri_r_base = preprocess_fmri(fmri_right_raw, timing_df, TR=TR, bin_sec=bin_sec, delay_sec=delay_sec)

            unique_runs = timing_df['run_id'].unique()
            video_bins = (timing_df['duration_sec'] / bin_sec).astype(int).values

            for normalization in use_normalization:
                
                # 2. CREATE WORKING COPIES
                fmri_l_proc = fmri_l_base.copy()
                fmri_r_proc = fmri_r_base.copy()
                
                # 3. PERFORM PER-RUN NORMALIZATION OR DEMEANING
                curr_bin_idx = 0 
                for rid in unique_runs:
                    v_indices = timing_df.index[timing_df['run_id'] == rid].tolist()
                    run_bin_length = sum(video_bins[v_indices])
                    run_slice = slice(curr_bin_idx, curr_bin_idx + run_bin_length)
                    
                    if normalization:
                        fmri_l_proc[valid_mask_left, run_slice] = zscore(fmri_l_proc[valid_mask_left, run_slice], axis=1)
                        fmri_r_proc[valid_mask_right, run_slice] = zscore(fmri_r_proc[valid_mask_right, run_slice], axis=1)
                    else:
                        fmri_l_proc[valid_mask_left, run_slice] -= fmri_l_proc[valid_mask_left, run_slice].mean(axis=1, keepdims=True)
                        fmri_r_proc[valid_mask_right, run_slice] -= fmri_r_proc[valid_mask_right, run_slice].mean(axis=1, keepdims=True)
                    
                    curr_bin_idx += run_bin_length
                
                fmri_l_proc = np.nan_to_num(fmri_l_proc)
                fmri_r_proc = np.nan_to_num(fmri_r_proc)
                
                # --------------------------------------------------------
                # 4. Loop BlockDiag / Models / Modalities
                # --------------------------------------------------------
                for blockdiag in BLOCKDIAG:
                    base_lbl = f"k{n_neighbors}_{'normalized' if normalization else 'notnormalized'}{'_hrf' if hrf else ''}_{'blockdiag' if blockdiag else 'global'}"
                    if not hrf:
                        base_lbl += f"_delay{int(delay_sec)}s"
                    
                    for model_label in models:
                        for mod_name, mod_suffix in modalities.items():
                            current_run_label = f"{model_label}_{mod_name}"
                            file_prefix = model_label.replace('pe-av', 'peav').replace('-', '_').replace('_frame', '-frame')
                            
                            model_data_path = EMBEDDINGS_BASE / model_label / f"{int(bin_sec)}s" / f"{file_prefix}_{mod_suffix}.npy"
                            output_dir = RSA_RESULTS_BASE / f"{model_label}" / f"{base_lbl}" / f"{int(bin_sec)}s" / mod_name
                            
                            if not model_data_path.exists():
                                print(f"  [SKIP] Model missing: {model_data_path}")
                                continue

                            print(f"  -> RSA: {model_label} [{mod_name}] | {base_lbl}")
                            
                            for rsa_method in RSA_METHODS:
                                left_corr = complete_searchlight_rsa_pipeline(
                                    hemisphere='left',
                                    label=current_run_label,
                                    surface_file=surface_left,
                                    fmri_data=fmri_l_proc,
                                    timing_df=timing_df,
                                    bin_sec=bin_sec,
                                    model_data=str(model_data_path),
                                    output_dir=str(output_dir / "left_hemisphere"),
                                    data_dir=Path(data_dir), n_neighbors=n_neighbors,
                                    normalization=normalization,
                                    hrf=hrf,
                                    method=rsa_method,
                                    block_diagonal=blockdiag,
                                    limit_mm=None  
                                )

                                right_corr = complete_searchlight_rsa_pipeline(
                                    hemisphere='right',
                                    label=current_run_label,
                                    surface_file=surface_right,
                                    fmri_data=fmri_r_proc,
                                    timing_df=timing_df,
                                    bin_sec=bin_sec,
                                    model_data=str(model_data_path),
                                    output_dir=str(output_dir / "right_hemisphere"),
                                    data_dir=Path(data_dir), n_neighbors=n_neighbors,
                                    normalization=normalization,
                                    hrf=hrf,
                                    method=rsa_method,
                                    block_diagonal=blockdiag,
                                    limit_mm=None
                                )

                                print(f"\n  [CIFTI] Generating direct CIFTI using template header...")
                                    
                                # Use the correlation arrays
                                # Use the correlation arrays and clean NaNs
                                left_clean = np.nan_to_num(left_corr, nan=0.0)
                                right_clean = np.nan_to_num(right_corr, nan=0.0)
                                
                                # DROP MEDIAL WALL BEFORE SAVING CIFTI
                                left_masked = left_clean[valid_mask_left]
                                right_masked = right_clean[valid_mask_right]
                                
                                # Combine the masked data (Now perfectly matches 59412 template)
                                combined_data = np.hstack((left_masked, right_masked)).reshape(1, -1)
                                
                                template_img = nib.load(template_cifti_path)
                                
                                if combined_data.shape[1] != template_img.shape[1]:
                                    raise ValueError(
                                        f"Shape mismatch! Concatenated data: {combined_data.shape[1]} vertices, "
                                        f"Template: {template_img.shape[1]}."
                                    )

                                cifti = nib.Cifti2Image(
                                    combined_data.astype(np.float32), 
                                    header=template_img.header,
                                    nifti_header=template_img.nifti_header
                                )
                                
                                # Define and update metric name
                                metric_name = f"RSA_searchlight_{rsa_method}_{'blockdiag' if blockdiag else 'global'}"
                                try:
                                    scalar_axis = cifti.header.get_axis(0)
                                    scalar_axis.name[0] = metric_name
                                except Exception as e:
                                    print(f"    (Note: Could not update internal map name: {e})")

                                # Fix Output Path (Create dir + add filename)
                                cifti_dir = RSA_RESULTS_BASE / 'cifti_maps'
                                cifti_dir.mkdir(parents=True, exist_ok=True)
                                map_output = cifti_dir / f"rsa_{current_run_label}_searchlight_{base_lbl}_{rsa_method}_{int(bin_sec)}s.dscalar.nii"

                                nib.save(cifti, map_output)
                                print(f"  [SUCCESS] Saved CIFTI map: {map_output}")

print("\n\n ALL CONFIGURATIONS PROCESSED SUCCESSFULLY!")

LOADING RAW DATA (Running Once)
Loading Left Hemisphere...
Loading Right Hemisphere...
Data Loaded. Left Valid: 29696, Right Valid: 29716

[FMRI] Preprocessing: Bin=2.0s, Delay=0s, HRF=True
  -> RSA: pe-av-small-16-frame [joint] | k100_normalized_hrf_blockdiag
SEARCHLIGHT RSA PIPELINE [LEFT] - BLOCK-DIAGONAL (Within-Video)

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_left.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100 nearest neighbors...
Checking for 100-nearest neighbors file...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/neighbor_indices_k100_left.npy
  [FOUND] Neighbors file exists. Loading from disk...

STEP 3: Computing RSA correlation map...
fMRI data shape: 32492 vertices × 1581 timepoints
Performing Per-Run Normalization (Z-score)...

--- STRATEGY: BLOCK-DIAGONAL (WITHIN-VIDEO) 

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.5s
[Parallel(n_jobs=16)]: Done 480 tasks      | elapsed:    0.8s
[Parallel(n_jobs=16)]: Done 7760 tasks      | elapsed:    9.0s
[Parallel(n_jobs=16)]: Done 18960 tasks      | elapsed:   22.4s
[Parallel(n_jobs=16)]: Done 31934 tasks      | elapsed:   36.9s
[Parallel(n_jobs=16)]: Done 32492 out of 32492 | elapsed:   37.3s finished




RSA computation complete!
  Successfully computed: 30822 / 32492 (94.9%)
  Skipped (low variance): 1670 (5.1%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_normalized_hrf_blockdiag/2s/joint/left_hemisphere
SEARCHLIGHT RSA PIPELINE [RIGHT] - BLOCK-DIAGONAL (Within-Video)

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_right.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100 nearest neighbors...
Checking for 100-nearest neighbors file...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/neighbor_indices_k100_right.npy
  [FOUND] Neighbors file exists. Loading from disk...

STEP 3: Computing RSA correlation map...
fMRI data shape: 32492 vertices × 1581 timepoints
Performing 

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 1248 tasks      | elapsed:    1.0s
[Parallel(n_jobs=16)]: Done 9248 tasks      | elapsed:   10.6s
[Parallel(n_jobs=16)]: Done 20448 tasks      | elapsed:   24.0s
[Parallel(n_jobs=16)]: Done 32089 tasks      | elapsed:   37.1s
[Parallel(n_jobs=16)]: Done 32492 out of 32492 | elapsed:   37.4s finished
pixdim[1,2,3] should be non-zero; setting 0 dims to 1




RSA computation complete!
  Successfully computed: 30879 / 32492 (95.0%)
  Skipped (low variance): 1613 (5.0%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_normalized_hrf_blockdiag/2s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  [SUCCESS] Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_searchlight_k100_normalized_hrf_blockdiag_spearman_2s.dscalar.nii
  -> RSA: pe-av-small-16-frame [joint] | k100_normalized_hrf_global
SEARCHLIGHT RSA PIPELINE [LEFT] - GLOBAL

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_left.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100 nearest nei

...



RSA computation complete!
  Successfully computed: 30879 / 32492 (95.0%)
  Skipped (low variance): 1613 (5.0%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_normalized_blockdiag_delay5s/1s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  [SUCCESS] Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_searchlight_k100_normalized_blockdiag_delay5s_spearman_1s.dscalar.nii
  -> RSA: pe-av-small-16-frame [joint] | k100_normalized_global_delay5s
SEARCHLIGHT RSA PIPELINE [LEFT] - GLOBAL

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_left.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    4.8s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:   29.0s
[Parallel(n_jobs=16)]: Done 490 tasks      | elapsed:   53.3s
[Parallel(n_jobs=16)]: Done 938 tasks      | elapsed:  1.4min
[Parallel(n_jobs=16)]: Done 1394 tasks      | elapsed:  2.5min
[Parallel(n_jobs=16)]: Done 1944 tasks      | elapsed:  4.9min
[Parallel(n_jobs=16)]: Done 2594 tasks      | elapsed:  7.8min
[Parallel(n_jobs=16)]: Done 3344 tasks      | elapsed: 10.0min
[Parallel(n_jobs=16)]: Done 4194 tasks      | elapsed: 13.6min
[Parallel(n_jobs=16)]: Done 5144 tasks      | elapsed: 17.8min
[Parallel(n_jobs=16)]: Done 6194 tasks      | elapsed: 22.5min
[Parallel(n_jobs=16)]: Done 7344 tasks      | elapsed: 27.5min
[Parallel(n_jobs=16)]: Done 8594 tasks      | elapsed: 33.0min
[Parallel(n_jobs=16)]: Done 9944 tasks      | elapsed: 38.8min
[Parallel(n_jobs=16)]: Done 11394 tasks      



RSA computation complete!
  Successfully computed: 30822 / 32492 (94.9%)
  Skipped (low variance): 1670 (5.1%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_normalized_global_delay5s/1s/joint/left_hemisphere
SEARCHLIGHT RSA PIPELINE [RIGHT] - GLOBAL

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_right.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100 nearest neighbors...
Checking for 100-nearest neighbors file...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/neighbor_indices_k100_right.npy
  [FOUND] Neighbors file exists. Loading from disk...

STEP 3: Computing RSA correlation map...
fMRI data shape: 32492 vertices × 3162 timepoints
Performing Per-Run Normalization 

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    4.8s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:   29.1s
[Parallel(n_jobs=16)]: Done 491 tasks      | elapsed:   53.4s
[Parallel(n_jobs=16)]: Done 918 tasks      | elapsed:  1.3min
[Parallel(n_jobs=16)]: Done 1378 tasks      | elapsed:  2.5min
[Parallel(n_jobs=16)]: Done 1928 tasks      | elapsed:  4.9min
[Parallel(n_jobs=16)]: Done 2578 tasks      | elapsed:  7.8min
[Parallel(n_jobs=16)]: Done 3328 tasks      | elapsed: 10.0min
[Parallel(n_jobs=16)]: Done 4306 tasks      | elapsed: 14.2min
[Parallel(n_jobs=16)]: Done 5256 tasks      | elapsed: 18.3min
[Parallel(n_jobs=16)]: Done 6306 tasks      | elapsed: 23.0min
[Parallel(n_jobs=16)]: Done 7456 tasks      | elapsed: 28.1min
[Parallel(n_jobs=16)]: Done 8706 tasks      | elapsed: 33.7min
[Parallel(n_jobs=16)]: Done 10056 tasks      | elapsed: 39.7min


KeyboardInterrupt: 

In [6]:

import numpy as np
import pandas as pd
import scipy.io as sio
from pathlib import Path
import sys

# ============================================================================
# CONFIGURATION
# ============================================================================

''' 
Based on viewing the results, this configuration was chosen. de-meaning was added to when normalization is not performed. as a result the un-normalized results 
of configs other than 5s delay which didnt use normalization are not the latest version of the code 
(previous version used nothing when normalization is false; current one does demeaning). so the the only cofigurations with the latest variations in code are the ones
who are saved as cifti.
'''

use_normalization = [False]       # Target 'notnormalized'
use_hrf = [False]                 # Target 'delay5s'
BLOCKDIAG = [True, False]         # Run both 'blockdiag' and 'global'
BIN_SEC_LIST = [1.0]              # Target '1s'
DELAY_SEC_LIST = [5.0]            # Target 'delay5s'
n_neighbors = 100

# Paths
template_cifti_path = '/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.curvature_MSMAll.32k_fs_LR.dscalar.nii'
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output")
MAT_FILE_LEFT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
MAT_FILE_RIGHT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"
GLASSER_LEFT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii"
GLASSER_RIGHT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii"
TIMING_CSV = "/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv"

# Models & Modalities
models = {
    "pe-av-small-16-frame"
}
modalities = {
    "joint": "av",
    # "audio": "a",
    # "video": "v"
}
RSA_METHODS = ['spearman']
TR = 1.0  # Assumed TR 

# ============================================================================
# STEP 0: LOAD RAW DATA ONCE (Optimization)
# ============================================================================
print("=" * 70)
print("LOADING RAW DATA (Running Once)")
print("=" * 70)

timing_df = pd.read_csv(TIMING_CSV)

# Load Left fMRI
print("Loading Left Hemisphere...")
mat_data_left = sio.loadmat(MAT_FILE_LEFT)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492: fmri_left_raw = fmri_left_raw.T

# Load Right fMRI
print("Loading Right Hemisphere...")
mat_data_right = sio.loadmat(MAT_FILE_RIGHT)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492: fmri_right_raw = fmri_right_raw.T

# Pre-calculate valid masks (Medial Wall Removal)
valid_mask_left = fmri_left_raw.std(axis=1) > 0
valid_mask_right = fmri_right_raw.std(axis=1) > 0

print(f"Data Loaded. Left Valid: {np.sum(valid_mask_left)}, Right Valid: {np.sum(valid_mask_right)}")
data_dir = Path("/home/amin/Research/Representation/Movie/data/Setareh")
surface_left =  Path("/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii")
surface_right = Path("/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii")

# ============================================================================
# MAIN LOOP
# ============================================================================

for hrf in use_hrf:
    # If HRF is ON, Delay is irrelevant (set to 0). If HRF is OFF, loop over delays.
    current_delays = [0] if hrf else DELAY_SEC_LIST
    
    for delay_sec in current_delays:
        for bin_sec in BIN_SEC_LIST:
            
            # Print once per timescale
            print(f"\n[FMRI] Preprocessing: Bin={bin_sec}s, Delay={delay_sec}s, HRF={hrf}")
            
            # 1. LOAD RAW PREPROCESSED DATA (Base Copy)
            fmri_l_base = preprocess_fmri(fmri_left_raw, timing_df, TR=TR, bin_sec=bin_sec, delay_sec=delay_sec)
            fmri_r_base = preprocess_fmri(fmri_right_raw, timing_df, TR=TR, bin_sec=bin_sec, delay_sec=delay_sec)

            unique_runs = timing_df['run_id'].unique()
            video_bins = (timing_df['duration_sec'] / bin_sec).astype(int).values

            for normalization in use_normalization:
                
                # 2. CREATE WORKING COPIES
                fmri_l_proc = fmri_l_base.copy()
                fmri_r_proc = fmri_r_base.copy()
                
                # 3. PERFORM PER-RUN NORMALIZATION OR DEMEANING
                curr_bin_idx = 0 
                for rid in unique_runs:
                    v_indices = timing_df.index[timing_df['run_id'] == rid].tolist()
                    run_bin_length = sum(video_bins[v_indices])
                    run_slice = slice(curr_bin_idx, curr_bin_idx + run_bin_length)
                    
                    if normalization:
                        fmri_l_proc[valid_mask_left, run_slice] = zscore(fmri_l_proc[valid_mask_left, run_slice], axis=1)
                        fmri_r_proc[valid_mask_right, run_slice] = zscore(fmri_r_proc[valid_mask_right, run_slice], axis=1)
                    else:
                        fmri_l_proc[valid_mask_left, run_slice] -= fmri_l_proc[valid_mask_left, run_slice].mean(axis=1, keepdims=True)
                        fmri_r_proc[valid_mask_right, run_slice] -= fmri_r_proc[valid_mask_right, run_slice].mean(axis=1, keepdims=True)
                    
                    curr_bin_idx += run_bin_length
                
                fmri_l_proc = np.nan_to_num(fmri_l_proc)
                fmri_r_proc = np.nan_to_num(fmri_r_proc)
                
                # --------------------------------------------------------
                # 4. Loop BlockDiag / Models / Modalities
                # --------------------------------------------------------
                for blockdiag in BLOCKDIAG:
                    base_lbl = f"k{n_neighbors}_{'normalized' if normalization else 'notnormalized'}{'_hrf' if hrf else ''}_{'blockdiag' if blockdiag else 'global'}"
                    if not hrf:
                        base_lbl += f"_delay{int(delay_sec)}s"
                    
                    for model_label in models:
                        for mod_name, mod_suffix in modalities.items():
                            current_run_label = f"{model_label}_{mod_name}"
                            file_prefix = model_label.replace('pe-av', 'peav').replace('-', '_').replace('_frame', '-frame')
                            
                            model_data_path = EMBEDDINGS_BASE / model_label / f"{int(bin_sec)}s" / f"{file_prefix}_{mod_suffix}.npy"
                            output_dir = RSA_RESULTS_BASE / f"{model_label}" / f"{base_lbl}" / f"{int(bin_sec)}s" / mod_name
                            
                            if not model_data_path.exists():
                                print(f"  [SKIP] Model missing: {model_data_path}")
                                continue

                            print(f"  -> RSA: {model_label} [{mod_name}] | {base_lbl}")
                            
                            for rsa_method in RSA_METHODS:
                                left_corr = complete_searchlight_rsa_pipeline(
                                    hemisphere='left',
                                    label=current_run_label,
                                    surface_file=surface_left,
                                    fmri_data=fmri_l_proc,
                                    timing_df=timing_df,
                                    bin_sec=bin_sec,
                                    model_data=str(model_data_path),
                                    output_dir=str(output_dir / "left_hemisphere"),
                                    data_dir=Path(data_dir), n_neighbors=n_neighbors,
                                    normalization=normalization,
                                    hrf=hrf,
                                    method=rsa_method,
                                    block_diagonal=blockdiag,
                                    limit_mm=None  
                                )

                                right_corr = complete_searchlight_rsa_pipeline(
                                    hemisphere='right',
                                    label=current_run_label,
                                    surface_file=surface_right,
                                    fmri_data=fmri_r_proc,
                                    timing_df=timing_df,
                                    bin_sec=bin_sec,
                                    model_data=str(model_data_path),
                                    output_dir=str(output_dir / "right_hemisphere"),
                                    data_dir=Path(data_dir), n_neighbors=n_neighbors,
                                    normalization=normalization,
                                    hrf=hrf,
                                    method=rsa_method,
                                    block_diagonal=blockdiag,
                                    limit_mm=None
                                )

                                print(f"\n  [CIFTI] Generating direct CIFTI using template header...")
                                    
                                # Use the correlation arrays
                                # Use the correlation arrays and clean NaNs
                                left_clean = np.nan_to_num(left_corr, nan=0.0)
                                right_clean = np.nan_to_num(right_corr, nan=0.0)
                                
                                # DROP MEDIAL WALL BEFORE SAVING CIFTI
                                left_masked = left_clean[valid_mask_left]
                                right_masked = right_clean[valid_mask_right]
                                
                                # Combine the masked data (Now perfectly matches 59412 template)
                                combined_data = np.hstack((left_masked, right_masked)).reshape(1, -1)
                                
                                template_img = nib.load(template_cifti_path)
                                
                                if combined_data.shape[1] != template_img.shape[1]:
                                    raise ValueError(
                                        f"Shape mismatch! Concatenated data: {combined_data.shape[1]} vertices, "
                                        f"Template: {template_img.shape[1]}."
                                    )

                                cifti = nib.Cifti2Image(
                                    combined_data.astype(np.float32), 
                                    header=template_img.header,
                                    nifti_header=template_img.nifti_header
                                )
                                
                                # Define and update metric name
                                metric_name = f"RSA_searchlight_{rsa_method}_{'blockdiag' if blockdiag else 'global'}"
                                try:
                                    scalar_axis = cifti.header.get_axis(0)
                                    scalar_axis.name[0] = metric_name
                                except Exception as e:
                                    print(f"    (Note: Could not update internal map name: {e})")

                                # Fix Output Path (Create dir + add filename)
                                cifti_dir = RSA_RESULTS_BASE / 'cifti_maps'
                                cifti_dir.mkdir(parents=True, exist_ok=True)
                                map_output = cifti_dir / f"rsa_{current_run_label}_searchlight_{base_lbl}_{rsa_method}_{int(bin_sec)}s.dscalar.nii"

                                nib.save(cifti, map_output)
                                print(f"  [SUCCESS] Saved CIFTI map: {map_output}")

print("\n\n ALL CONFIGURATIONS PROCESSED SUCCESSFULLY!")

LOADING RAW DATA (Running Once)
Loading Left Hemisphere...
Loading Right Hemisphere...
Data Loaded. Left Valid: 29696, Right Valid: 29716

[FMRI] Preprocessing: Bin=1.0s, Delay=5.0s, HRF=False
  -> RSA: pe-av-small-16-frame [joint] | k100_notnormalized_blockdiag_delay5s
SEARCHLIGHT RSA PIPELINE [LEFT] - BLOCK-DIAGONAL (Within-Video)

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_left.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100 nearest neighbors...
Checking for 100-nearest neighbors file...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/neighbor_indices_k100_left.npy
  [FOUND] Neighbors file exists. Loading from disk...

STEP 3: Computing RSA correlation map...
fMRI data shape: 32492 vertices × 3162 timepoints
Performing Per-Run Demeaning (Mean centering)...

--- STRATEGY: BLOCK-DIAGONAL (W

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.8s
[Parallel(n_jobs=16)]: Done 680 tasks      | elapsed:    2.6s
[Parallel(n_jobs=16)]: Done 3052 tasks      | elapsed:   17.7s
[Parallel(n_jobs=16)]: Done 4160 tasks      | elapsed:   25.0s
[Parallel(n_jobs=16)]: Done 4610 tasks      | elapsed:   28.3s
[Parallel(n_jobs=16)]: Done 5160 tasks      | elapsed:   32.4s
[Parallel(n_jobs=16)]: Done 5810 tasks      | elapsed:   37.3s
[Parallel(n_jobs=16)]: Done 6560 tasks      | elapsed:   43.4s
[Parallel(n_jobs=16)]: Done 7410 tasks      | elapsed:   50.2s
[Parallel(n_jobs=16)]: Done 8360 tasks      | elapsed:   57.9s
[Parallel(n_jobs=16)]: Done 9410 tasks      | elapsed:  1.1min
[Parallel(n_jobs=16)]: Done 10560 tasks      | elapsed:  1.3min
[Parallel(n_jobs=16)]: Done 11810 tasks      | elapsed:  1.4min
[Parallel(n_jobs=16)]: Done 13160 tasks      | elapsed:  1.6min
[Parallel(n_jobs=16)]: Done 14610 tasks 



RSA computation complete!
  Successfully computed: 30822 / 32492 (94.9%)
  Skipped (low variance): 1670 (5.1%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_notnormalized_blockdiag_delay5s/1s/joint/left_hemisphere
SEARCHLIGHT RSA PIPELINE [RIGHT] - BLOCK-DIAGONAL (Within-Video)

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_right.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100 nearest neighbors...
Checking for 100-nearest neighbors file...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/neighbor_indices_k100_right.npy
  [FOUND] Neighbors file exists. Loading from disk...

STEP 3: Computing RSA correlation map...
fMRI data shape: 32492 vertices × 3162 timepoints
Perf

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.4s
[Parallel(n_jobs=16)]: Done 576 tasks      | elapsed:    2.0s
[Parallel(n_jobs=16)]: Done 1986 tasks      | elapsed:    9.8s
[Parallel(n_jobs=16)]: Done 2336 tasks      | elapsed:   12.8s
[Parallel(n_jobs=16)]: Done 2786 tasks      | elapsed:   16.2s
[Parallel(n_jobs=16)]: Done 4472 tasks      | elapsed:   27.0s
[Parallel(n_jobs=16)]: Done 5122 tasks      | elapsed:   31.8s
[Parallel(n_jobs=16)]: Done 5872 tasks      | elapsed:   37.5s
[Parallel(n_jobs=16)]: Done 6722 tasks      | elapsed:   43.8s
[Parallel(n_jobs=16)]: Done 7672 tasks      | elapsed:   51.1s
[Parallel(n_jobs=16)]: Done 8722 tasks      | elapsed:   59.3s
[Parallel(n_jobs=16)]: Done 9872 tasks      | elapsed:  1.1min
[Parallel(n_jobs=16)]: Done 11122 tasks      | elapsed:  1.3min
[Parallel(n_jobs=16)]: Done 12472 tasks      | elapsed:  1.5min
[Parallel(n_jobs=16)]: Done 13922 tasks  



RSA computation complete!
  Successfully computed: 30879 / 32492 (95.0%)
  Skipped (low variance): 1613 (5.0%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_notnormalized_blockdiag_delay5s/1s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  [SUCCESS] Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_searchlight_k100_notnormalized_blockdiag_delay5s_spearman_1s.dscalar.nii
  -> RSA: pe-av-small-16-frame [joint] | k100_notnormalized_global_delay5s
SEARCHLIGHT RSA PIPELINE [LEFT] - GLOBAL

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_left.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Che

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    4.8s
[Parallel(n_jobs=16)]: Done 200 tasks      | elapsed:   32.8s
[Parallel(n_jobs=16)]: Done 497 tasks      | elapsed:   53.4s
[Parallel(n_jobs=16)]: Done 938 tasks      | elapsed:  1.4min
[Parallel(n_jobs=16)]: Done 1394 tasks      | elapsed:  2.6min
[Parallel(n_jobs=16)]: Done 1944 tasks      | elapsed:  5.1min
[Parallel(n_jobs=16)]: Done 2594 tasks      | elapsed:  8.0min
[Parallel(n_jobs=16)]: Done 3344 tasks      | elapsed: 10.3min
[Parallel(n_jobs=16)]: Done 4194 tasks      | elapsed: 14.0min
[Parallel(n_jobs=16)]: Done 5144 tasks      | elapsed: 18.3min
[Parallel(n_jobs=16)]: Done 6194 tasks      | elapsed: 22.9min
[Parallel(n_jobs=16)]: Done 7344 tasks      | elapsed: 28.0min
[Parallel(n_jobs=16)]: Done 8594 tasks      | elapsed: 33.6min
[Parallel(n_jobs=16)]: Done 9944 tasks      | elapsed: 39.5min
[Parallel(n_jobs=16)]: Done 11394 tasks      



RSA computation complete!
  Successfully computed: 30822 / 32492 (94.9%)
  Skipped (low variance): 1670 (5.1%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_notnormalized_global_delay5s/1s/joint/left_hemisphere
SEARCHLIGHT RSA PIPELINE [RIGHT] - GLOBAL

STEP 1: Checking geodesic distances...
Checking for geodesic distance matrix...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/geodesic_distances.dconn_right.nii
  [FOUND] Distance matrix already exists. Skipping computation.

STEP 2: Checking 100 nearest neighbors...
Checking for 100-nearest neighbors file...
  Target Output: /home/amin/Research/Representation/Movie/data/Setareh/neighbor_indices_k100_right.npy
  [FOUND] Neighbors file exists. Loading from disk...

STEP 3: Computing RSA correlation map...
fMRI data shape: 32492 vertices × 3162 timepoints
Performing Per-Run Demeaning (

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    4.8s
[Parallel(n_jobs=16)]: Done 200 tasks      | elapsed:   35.6s
[Parallel(n_jobs=16)]: Done 497 tasks      | elapsed:   56.1s
[Parallel(n_jobs=16)]: Done 998 tasks      | elapsed:  1.5min
[Parallel(n_jobs=16)]: Done 1490 tasks      | elapsed:  3.1min
[Parallel(n_jobs=16)]: Done 2040 tasks      | elapsed:  5.6min
[Parallel(n_jobs=16)]: Done 2690 tasks      | elapsed:  8.5min
[Parallel(n_jobs=16)]: Done 3472 tasks      | elapsed: 10.8min
[Parallel(n_jobs=16)]: Done 4322 tasks      | elapsed: 14.4min
[Parallel(n_jobs=16)]: Done 5272 tasks      | elapsed: 18.6min
[Parallel(n_jobs=16)]: Done 6322 tasks      | elapsed: 23.1min
[Parallel(n_jobs=16)]: Done 7472 tasks      | elapsed: 28.3min
[Parallel(n_jobs=16)]: Done 8722 tasks      | elapsed: 34.0min
[Parallel(n_jobs=16)]: Done 10072 tasks      | elapsed: 40.1min
[Parallel(n_jobs=16)]: Done 11522 tasks     



RSA computation complete!
  Successfully computed: 30879 / 32492 (95.0%)
  Skipped (low variance): 1613 (5.0%)
  Failed (errors): 0 (0.0%)

PIPELINE COMPLETE!
None-Image Results saved in: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/pe-av-small-16-frame/k100_notnormalized_global_delay5s/1s/joint/right_hemisphere

  [CIFTI] Generating direct CIFTI using template header...
  [SUCCESS] Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/cifti_maps/rsa_pe-av-small-16-frame_joint_searchlight_k100_notnormalized_global_delay5s_spearman_1s.dscalar.nii


 ALL CONFIGURATIONS PROCESSED SUCCESSFULLY!


# Misc

## bin 2s normalized_hrf

In [ ]:
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)
# Input fMRI files
mat_file_left = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
mat_file_right = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"

# Timing file
timing_csv = "/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv"
# Parameters

TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 0.0

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T
#remove the medial wall
row_std_left = fmri_left_raw.std(axis=1)
valid_mask_left = row_std_left > 0

print(f"Raw fMRI without the medial wall shape: {np.sum(valid_mask_left)}")
fmri_l_processed = preprocess_fmri(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC)
fmri_l_processed[valid_mask_left] = (
    fmri_l_processed[valid_mask_left]- fmri_l_processed[valid_mask_left].mean(axis=1, keepdims=True)) / fmri_l_processed[valid_mask_left].std(axis=1, keepdims=True)
print(f"Processed left fMRI shape: {fmri_l_processed.shape}")
# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T
#remove the medial wall
row_std_right = fmri_right_raw.std(axis=1)
valid_mask_right = row_std_right > 0
print(f"Raw fMRI without the medial wall shape: {np.sum(valid_mask_right)}")
fmri_r_processed = preprocess_fmri(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
fmri_r_processed[valid_mask_right] = (
    fmri_r_processed[valid_mask_right]- fmri_r_processed[valid_mask_right].mean(axis=1, keepdims=True)) / fmri_r_processed[valid_mask_right].std(axis=1, keepdims=True)
print(f"Processed right fMRI shape: {fmri_r_processed.shape}")

## pe-av

In [ ]:
# ============================================================================
# AUTOMATED MULTI-MODEL RSA LOOP
# ============================================================================

import sys
from pathlib import Path

# --- 1. CONFIGURATION ---

# Paths to Input Data
SURFACE_DIR = "/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1"

# Surface Files
surface_left = SURFACE_DIR / "S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"
surface_right = SURFACE_DIR / "S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii"

# Parameters
preprocessing_label = 'normalized_hrf'
normalization=True
hrf=True
N_NEIGHBORS = 100
LIMIT_MM = None
RSA_METHOD = 'spearman'

# Model Definitions
models = {
    # "pe-av-base": "pe-av-base",
    # "pe-av-base-16-frame": "pe-av-base-16-frame",
    # "pe-av-large": "pe-av-large",
    # "pe-av-large-16-frame": "pe-av-large-16-frame",
    # "pe-av-small": "pe-av-small",
    "pe-av-small-16-frame": "pe-av-small-16-frame"
}

modalities = {
    "joint": "av",
    # "audio": "a",
    # "video": "v"
}

# Base Paths
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
RSA_RESULTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output")

# --- 2. EXECUTION LOOP ---

for folder_name, model_label in models.items():
    for mod_name, mod_suffix in modalities.items():
        
        # Construct the specific label for this run
        current_run_label = f"{model_label}_{mod_name}"
        
        # Path to the specific .npy embedding file
        # Logic: folder 'pe-av-small-16-frame' -> file 'peav_small_16frame_av.npy'
        file_prefix = folder_name.replace('pe-av', 'peav').replace('-', '_').replace('_frame', 'frame')
        model_data_path = EMBEDDINGS_BASE / folder_name / f"{file_prefix}_{mod_suffix}.npy"
        
        if not model_data_path.exists():
            print(f"SKIPPING: File not found -> {model_data_path}")
            continue

        # Define output directory
        # e.g., .../pe-av-small-16-frame_canonicalhrf_bin2s/joint
        output_dir = RSA_RESULTS_BASE / f"{model_label}_{preprocessing_label}_bin{BIN_SEC}s" / mod_name
        
        print(f"\n\n{'='*60}")
        print(f"STARTING SEARCHLIGHT RSA: {model_label} [{mod_name.upper()}]")
        print(f"Model file: {model_data_path}")
        print(f"{'='*60}")

        # --- Process LEFT Hemisphere ---
        print("\n--> LEFT HEMISPHERE")
        complete_searchligt_rsa_pipeline(
            hemisphere='left',
            label=current_run_label,
            surface_file=str(surface_left),
            fmri_data=fmri_l_processed,  
            model_data=str(model_data_path),
            output_dir=str(output_dir / "left_hemisphere"),
            data_dir=str(DATA_DIR),         
            n_neighbors=N_NEIGHBORS,
            limit_mm=LIMIT_MM,normalization=normalization, hrf=hrf,
            method=RSA_METHOD
        )

        # --- Process RIGHT Hemisphere ---
        print("\n--> RIGHT HEMISPHERE")
        complete_searchligt_rsa_pipeline(
            hemisphere='right',
            label=current_run_label,
            surface_file=str(surface_right), 
            fmri_data=fmri_r_processed,  
            model_data=str(model_data_path),
            output_dir=str(output_dir / "right_hemisphere"),
            data_dir=str(DATA_DIR),         
            n_neighbors=N_NEIGHBORS,
            limit_mm=LIMIT_MM,normalization=normalization, hrf=hrf,
            method=RSA_METHOD
        )

print("\n\n ALL MODELS AND MODALITIES PROCESSED SUCCESSFULLY!")